In [ ]:
from diffusers import ControlNetModel, StableDiffusionControlNetPipeline
from diffusers.utils import load_image

# ControlNet is an adapter for Stable Diffusion that allows for precise structural control by adding extra conditions like edges, poses, or depth maps.
# While standard Stable Diffusion relies on text prompts, ControlNet forces the model to follow a specific geometry or composition provided by a reference image.

import torch
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import os

# get photo of bear
image_name = "/content/bear.jpg"
base_image = Image.open(image_name).convert("RGB")

# select model
model_id = "runwayml/stable-diffusion-v1-5"

# load pipeline
controlnet = ControlNetModel.from_pretrained("lllyasviel/sd-controlnet-canny", torch_dtype=torch.float32)

pipe = StableDiffusionControlNetPipeline.from_pretrained(
    model_id, controlnet=controlnet, torch_dtype=torch.float32
).to("cpu")

# use CPU
device = "cpu"
pipe = pipe.to(device)

# our prompt for CLIP
prompt = "A silhouette of a bear made of bright flowers, texture of tulips and roses,  hyperrealistic, 4k"

# function-callback which call in each step of scheduler
def latents_callback(step, timestamp, latents):
  print(f"Step {step}. Current value of noise: {latents.mean().item():.4f}")

# generation
print("starting of generation...")

generator = torch.Generator("cpu").manual_seed(27)

result_image = pipe(
    prompt=prompt,
    image=base_image,
    generator=generator,
    num_inference_steps=20, # num_inference_steps - scheduler for U-Net
    guidance_scale=7.5,
    controlnet_conditioning_scale=0.9, # control of base_image
    callback=latents_callback,
    callback_steps=1 # call 'latents_callback' each step
).images[0]

# save image
result_image.save("polar_bear.png")

plt.imshow(result_image)
plt.axis("off")
plt.show()

print("Finished!")